# ResNet50 Accident Detection — VS Code

This notebook trains **ResNet50** using transfer learning on your existing local dataset.

## Before running
1. Open this notebook from the project's `notebooks/` folder.
2. Select your project's `.venv` Python interpreter in VS Code.
3. Install dependencies using `pip install -r requirements.txt`.
4. Check `DATA_DIR` and `CLASS_NAMES` in the setup cell.
5. Run cells from top to bottom.

Class mapping is `non_accident = 0`, `accident = 1`. The folder names must match exactly. The test set is reserved for final evaluation.


In [3]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

MODEL_NAME = "ResNet50"
MODEL_CLASS = ResNet50
CHECKPOINT_FILENAME = "resnet50_best.keras"
HISTORY_FILENAME = "resnet50_history.csv"
CURVES_FILENAME = "resnet50_training_curves.png"
CM_FILENAME = "resnet50_confusion_matrix.png"
METRICS_FILENAME = "resnet50_metrics.csv"
PREDICTIONS_FILENAME = "resnet50_test_predictions.npz"


In [ ]:

from pathlib import Path

PROJECT_DIR = Path.cwd()

# Move to the project root if the notebook is inside notebooks/
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "datasets" / "data"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "results"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATA_DIR)

print("Train folder exists:", TRAIN_DIR.exists())
print("Validation folder exists:", VAL_DIR.exists())
print("Test folder exists:", TEST_DIR.exists())


SyntaxError: invalid syntax (2469853137.py, line 27)

In [ ]:
# Validate dataset folders and count image files
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

for folder in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    if not folder.is_dir():
        raise FileNotFoundError(
            f"Folder not found: {folder}. Update DATA_DIR in the setup cell."
        )
    found = sorted(p.name for p in folder.iterdir() if p.is_dir())
    if sorted(found) != sorted(CLASS_NAMES):
        raise ValueError(f"Expected {CLASS_NAMES} in {folder}; found {found}")

def count_images(folder):
    return {
        cls: sum(1 for p in (folder / cls).rglob("*")
                 if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS)
        for cls in CLASS_NAMES
    }

print("Train:", count_images(TRAIN_DIR))
print("Validation:", count_images(VAL_DIR))
print("Test:", count_images(TEST_DIR))

def load_dataset(folder, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        str(folder), class_names=CLASS_NAMES, label_mode="binary",
        image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        shuffle=shuffle, seed=SEED
    )

train_raw = load_dataset(TRAIN_DIR, True)
val_raw = load_dataset(VAL_DIR, False)
test_raw = load_dataset(TEST_DIR, False)
AUTOTUNE = tf.data.AUTOTUNE


In [ ]:
# Model-specific preprocessing (preprocess_input is imported in the first cell)
def preprocess_data(images, labels):
    return preprocess_input(tf.cast(images, tf.float32)), labels

train_ds = train_raw.map(preprocess_data, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_raw.map(preprocess_data, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_raw.map(preprocess_data, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Build pretrained CNN and binary classification head
base_model = MODEL_CLASS(
    weights="imagenet", include_top=False,
    input_shape=(224, 224, 3), pooling="avg"
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.Dropout(0.30)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = Model(inputs, outputs, name=MODEL_NAME + "_Accident")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)
model.summary()

# Train; keep the checkpoint selected using validation AUC
checkpoint_path = MODEL_DIR / CHECKPOINT_FILENAME
callbacks = [
    ModelCheckpoint(str(checkpoint_path), monitor="val_auc", mode="max",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True,
                  verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2,
                      min_lr=1e-6, verbose=1),
]

print(f"Starting {MODEL_NAME} training...")
start_time = time.time()
history = model.fit(train_ds, validation_data=val_ds,
                    epochs=EPOCHS, callbacks=callbacks)
training_seconds = time.time() - start_time
print(f"Training time: {training_seconds / 60:.2f} minutes")

pd.DataFrame(history.history).to_csv(
    RESULT_DIR / HISTORY_FILENAME, index_label="epoch"
)

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Train")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title(f"{MODEL_NAME} Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history["loss"], label="Train")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title(f"{MODEL_NAME} Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.savefig(RESULT_DIR / CURVES_FILENAME, dpi=150)
plt.show()

# Final evaluation on untouched test data
best_model = tf.keras.models.load_model(checkpoint_path)
y_true = np.concatenate([labels.numpy().reshape(-1)
                         for _, labels in test_ds]).astype(int)

start = time.time()
y_prob = best_model.predict(test_ds, verbose=1).reshape(-1)
prediction_seconds = time.time() - start
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "model": MODEL_NAME,
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1_score": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_prob),
    "test_images": len(y_true),
    "training_seconds": training_seconds,
    "test_prediction_seconds": prediction_seconds,
    "seconds_per_image": prediction_seconds / max(len(y_true), 1),
}

print(f"\\n{MODEL_NAME} — TEST RESULTS")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}" if isinstance(value, (float, np.floating))
          else f"{key}: {value}")

print("\\nClassification report:")
print(classification_report(y_true, y_pred, labels=[0, 1],
                            target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES).plot(values_format="d")
plt.title(f"{MODEL_NAME} Test Confusion Matrix")
plt.tight_layout()
plt.savefig(RESULT_DIR / CM_FILENAME, dpi=150)
plt.show()

pd.DataFrame([metrics]).to_csv(RESULT_DIR / METRICS_FILENAME, index=False)
np.savez(RESULT_DIR / PREDICTIONS_FILENAME,
         y_true=y_true, y_pred=y_pred, y_prob=y_prob)

print("\\nFinished.")
print("Best model:", checkpoint_path)
print("Results folder:", RESULT_DIR)


In [2]:
import sys
print(sys.executable)

import tensorflow as tf
print(tf.__version__)

c:\Users\steph\AppData\Local\Programs\Python\Python314\python.exe
2.22.0-rc0


In [5]:
print("Notebook is working!")
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))
print("Available CPUs:", tf.config.list_physical_devices("CPU"))

Notebook is working!
TensorFlow: 2.22.0-rc0
Available GPUs: []
Available CPUs: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [6]:
import time

start = time.time()

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Import time:", round(time.time() - start, 2), "seconds")

TensorFlow version: 2.22.0-rc0
Import time: 0.0 seconds


In [7]:
from pathlib import Path

DATA_DIR = Path.cwd() / "datasets" / "data"

for split in ["train", "val", "test"]:
    split_dir = DATA_DIR / split

    print(f"\n{split.upper()}")

    if split_dir.exists():
        for class_dir in sorted(split_dir.iterdir()):
            if class_dir.is_dir():
                count = sum(
                    1 for p in class_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in
                    [".jpg", ".jpeg", ".png", ".bmp"]
                )
                print(f"{class_dir.name}: {count} images")
    else:
        print(f"Folder not found: {split_dir}")


TRAIN
Folder not found: c:\Users\steph\OneDrive\Desktop\New folder\Accident-Detection-and-Alerting-\accident-detection\notebooks\datasets\data\train

VAL
Folder not found: c:\Users\steph\OneDrive\Desktop\New folder\Accident-Detection-and-Alerting-\accident-detection\notebooks\datasets\data\val

TEST
Folder not found: c:\Users\steph\OneDrive\Desktop\New folder\Accident-Detection-and-Alerting-\accident-detection\notebooks\datasets\data\test


## Optional fine-tuning experiment

First save and record the frozen-backbone baseline. For a separate experiment, you can unfreeze only the last backbone layers and train with a smaller learning rate (for example `1e-5`). Keep fine-tuning results separate from baseline results, and do not use the test set to tune hyperparameters.
